# 🟡 Interview: Implement Focal Loss

$$FL(p_t) = -\alpha (1-p_t)^\gamma \log(p_t)$$

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    """
    手写 Focal Loss —— 面试版。

    面试考点:
    1. Focal Loss = -alpha * (1-p_t)^gamma * log(p_t)
    2. (1-p_t)^gamma 是聚焦因子: 简单样本(p_t大)权重小, 困难样本权重大
    3. 手写 softmax + 数值稳定是必考
    4. 高级索引 probs[arange(N), targets] 取对角线

    参数: logits: (N, C), targets: (N,), alpha: float, gamma: float
    返回: scalar
    """
    N, C = logits.shape

    # ---- Step 1: 数值稳定 softmax ----
    # 减去最大值防溢出
    max_vals = logits.max(dim=-1, keepdim=True).values  # (N, 1)
    shifted = logits - max_vals  # (N, C)
    exp_s = torch.exp(shifted)    # (N, C)
    probs = exp_s / exp_s.sum(dim=-1, keepdim=True)  # (N, C)

    # ---- Step 2: 取正确类别的概率 ----
    # arange(N) 是行索引, targets 是列索引
    # 高级索引: 取出 probs[0,targets[0]], probs[1,targets[1]], ...
    p_t = probs[torch.arange(N), targets]  # (N,)

    # ---- Step 3: Focal Loss ----
    # (1-p_t)^gamma: 聚焦因子
    #   gamma=0 → 等价于 alpha * CE loss (无聚焦)
    #   gamma=2 → RetinaNet 默认值，对 p_t>0.5 的样本大幅降权
    log_p_t = torch.log(p_t + 1e-8)  # +1e-8 防 log(0)
    fl = -alpha * (1 - p_t) ** gamma * log_p_t

    return fl.mean()

In [ ]:
import torch.nn.functional as F

torch.manual_seed(0)
logits = torch.randn(8, 4)
targets = torch.randint(0, 4, (8,))

# gamma=0 should match weighted CE (alpha * CE)
fl_gamma0 = focal_loss(logits, targets, alpha=0.25, gamma=0.0)
ce = F.cross_entropy(logits, targets)
print(f"Focal (gamma=0): {fl_gamma0:.4f}  |  alpha * CE: {0.25 * ce:.4f}")

# Higher gamma should reduce loss on easy examples
fl_g2 = focal_loss(logits, targets, alpha=0.25, gamma=2.0)
fl_g5 = focal_loss(logits, targets, alpha=0.25, gamma=5.0)
print(f"Focal gamma=2: {fl_g2:.4f}  |  gamma=5: {fl_g5:.4f}  (higher gamma -> lower loss on easy examples)")

In [ ]:
from torch_judge import check
check("focal_loss")

## 📝 核心思路总结

1. **Focal Loss 的核心**：`(1-p_t)^gamma` 聚焦因子让简单样本（p_t 大）的损失趋近 0，迫使模型关注困难样本。
2. **gamma 的作用**：gamma=0 等价于加权 CE；gamma=2（默认）对 p_t>0.5 的样本大幅降权；gamma 越大聚焦效果越强。
3. **数值稳定 softmax**：先减最大值再 exp，是 PyTorch 内部 softmax 的标准实现，面试高频考点。
4. **高级索引取对角线**：`probs[arange(N), targets]` 是 PyTorch 中取出「每个样本对应类别概率」的标准技巧。